# Đánh giá cuối cùng B2 cho Member 4
Upload `final_test/` vào `MyDrive/LDTF_4_MEMBERS/`, chọn GPU T4 và Run all. B2 đã được chọn bằng validation. Notebook sẽ truy cập tập test khi chạy cell 5. Chạy lại dùng dự đoán hoàn tất đã lưu; nếu ngắt giữa suy luận, chạy lại từ đầu lượt suy luận và ghi thêm ledger.

## 1. Kết nối Drive và cài môi trường

In [ ]:
import os, sys, json, hashlib, zipfile, subprocess, threading, time
from pathlib import Path
from tqdm.auto import tqdm
RUN_TAG = "B2_colab_detailed_01"
BOOTSTRAP_SAMPLES = 1000
if not RUN_TAG or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in RUN_TAG):
    raise ValueError("RUN_TAG chỉ gồm chữ, số, gạch dưới hoặc gạch ngang.")
# Local simulation uses an isolated fixture folder; normal Colab mounts Drive.
simulation = os.environ.get("LDTF_COLAB_SIMULATION_ROOT")
with tqdm(total=4, desc="Chuẩn bị môi trường", unit="bước") as progress:
    progress.set_postfix_str("Kết nối Drive")
    if simulation:
        MEMBER_ROOT = Path(simulation)
    else:
        from google.colab import drive
        drive.mount("/content/drive")
        MEMBER_ROOT = Path("/content/drive/MyDrive/LDTF_4_MEMBERS")
    progress.update(1)
    progress.set_postfix_str("Kiểm tra source")
    source = MEMBER_ROOT / "final_test/source/LDTF_final_test_source.zip"
    manifest = json.loads((source.parent / "manifest.json").read_text())
    if hashlib.sha256(source.read_bytes()).hexdigest() != manifest["sha256"]:
        raise ValueError("Source ZIP sai checksum; upload lại final_test/source.")
    progress.update(1)
    progress.set_postfix_str("Giải nén source")
    import tempfile
    extraction = Path(tempfile.mkdtemp(prefix="ldtf_final_test_"))
    with zipfile.ZipFile(source) as archive:
        for item in tqdm(archive.infolist(), desc="Giải nén", unit="file", leave=False):
            target = (extraction / item.filename).resolve()
            if not target.is_relative_to(extraction.resolve()):
                raise ValueError("Đường dẫn ZIP không hợp lệ.")
            archive.extract(item, extraction)
    PROJECT = extraction / "LDTF_bert_source"
    progress.update(1)
    progress.set_postfix_str("Cài thư viện; xem log bên dưới")
    if not simulation:
        done = threading.Event()
        def heartbeat():
            started = time.monotonic()
            while not done.wait(1):
                progress.set_postfix_str(f"Đang cài thư viện | {time.monotonic()-started:.0f}s")
        worker = threading.Thread(target=heartbeat, daemon=True)
        worker.start()
        process = None
        try:
            process = subprocess.Popen([sys.executable, "-u", "-m", "pip", "install", "-r", str(PROJECT / "requirements.txt")],
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end="", flush=True)
            if process.wait():
                raise RuntimeError("Cài thư viện thất bại; xem log pip.")
        finally:
            if process is not None and process.poll() is None:
                process.terminate()
                process.wait()
            done.set(); worker.join()
    if str(PROJECT) not in sys.path:
        sys.path.insert(0, str(PROJECT))
    progress.update(1)
    progress.set_postfix_str("Hoàn tất")
from experiments.final_test_package import FinalTest
import torch
if not simulation and not torch.cuda.is_available():
    raise RuntimeError("Chọn Runtime > Change runtime type > GPU T4 rồi chạy lại.")
workflow = FinalTest(MEMBER_ROOT, overrides={
    "output_dir": f"final_test/colab_runs/{RUN_TAG}",
    "bundle_file": f"final_test/colab_runs/{RUN_TAG}_final_test.zip",
})
print("Thiết bị:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU mô phỏng")
print("Đầu ra:", workflow.out)

## 2. Kiểm tra dữ liệu và checksum

In [ ]:
workflow.validate()

## 3. Xác minh checkpoint B2

In [ ]:
workflow.check_checkpoint()

## 4. Chuẩn bị tokenizer và model

In [ ]:
workflow.prepare()

## 5. Chạy đánh giá chính thức trên test

In [ ]:
from experiments.final_eval import evaluate_package
evaluate_package(workflow)

## 6. Phân tích kết quả và vẽ biểu đồ

In [ ]:
metrics = workflow.analyze()
try:
    from IPython.display import display, Image
except ImportError:
    print('Biểu đồ đã lưu tại:', workflow.out / 'figures')
else:
    display(Image(filename=str(workflow.out / 'figures/confusion_matrix.png')))
    display(Image(filename=str(workflow.out / 'figures/per_class_f1.png')))

## 7. Phân tích chuyên sâu và khoảng tin cậy

In [ ]:
from experiments.local_test_report import extended_report
stats = extended_report(workflow, bootstrap_samples=BOOTSTRAP_SAMPLES)
print(json.dumps(stats, ensure_ascii=False, indent=2))
try:
    from IPython.display import display, Image
except ImportError:
    print("Biểu đồ:", workflow.out / "figures")
else:
    for name in ("confusion_matrix_normalized.png", "calibration.png"):
        display(Image(filename=str(workflow.out / "figures" / name)))

## 8. Xem chỉ số từng nhãn và các mẫu sai

In [ ]:
import pandas as pd
with tqdm(total=4, desc="Đọc bảng phân tích", unit="bảng") as progress:
    for name in ("metrics/per_class_metrics.csv", "metrics/confusion_pairs.csv",
                 "metrics/by_word_length.csv", "predictions/high_confidence_errors.csv"):
        progress.set_postfix_str(name)
        table = pd.read_csv(workflow.out / name)
        print("\n" + name)
        try:
            from IPython.display import display
        except ImportError:
            print(table.head(20).to_string(index=False))
        else:
            display(table.head(20))
        progress.update(1)
    progress.set_postfix_str("Hoàn tất")

## 9. Xuất toàn bộ báo cáo và ZIP bàn giao

In [ ]:
bundle = workflow.export()
print('Báo cáo chi tiết:', workflow.out / 'reports/DETAILED_REPORT.html')
print('ZIP đã lưu trên Drive:', bundle)